# Topic 16 - FastAPI Essentials

Build a small FastAPI service with typed routes, dependency injection, response models, and a passing TestClient test. All code by hand.

## Learning objectives

- create typed routes with path/query parameters
- validate request/response bodies with Pydantic models
- use `Depends()` for dependency injection
- set explicit status codes and `response_model`
- write TestClient tests without a running server
- verify automatic docs at `/docs` and `/openapi.json`

## 1. Create the FastAPI app

Install: `pip install fastapi httpx pytest`

The service models a tiny invoice tracker.

In [8]:
from typing import Annotated
from fastapi import FastAPI, Depends, HTTPException, status
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field

app = FastAPI(title="Mini Invoice Tracker")

invoices_db: dict[int, dict] = {}
next_id = 1

## 2. Pydantic models for request/response

In [9]:
class InvoiceBase(BaseModel):
    invoice_number: str = Field(min_length=1)
    vendor: str = Field(min_length=1)
    total: float = Field(gt=0)

class InvoiceCreate(InvoiceBase):
    """Input for creating an invoice."""
    pass

class InvoiceOut(InvoiceBase):
    """Output — adds id."""
    id: int

class InvoiceUpdate(BaseModel):
    """Partial update — all fields optional."""
    invoice_number: str | None = None
    vendor: str | None = None
    total: float | None = Field(default=None, gt=0)

## 3. Dependency: invoice store

A dependency that provides access to the in-memory store. `Depends()` injects it into routes.

In [10]:
def get_invoice_store():
    """Dependency that yields the invoice store."""
    return invoices_db

StoreDep = Annotated[dict, Depends(get_invoice_store)]

## 4. Routes

- `POST /invoices` — create (201)
- `GET /invoices/{invoice_id}` — read one (200 / 404)
- `GET /invoices` — list with query params
- `PATCH /invoices/{invoice_id}` — partial update
- `DELETE /invoices/{invoice_id}` — delete (204)

In [11]:
@app.post("/invoices", status_code=status.HTTP_201_CREATED, response_model=InvoiceOut)
async def create_invoice(
    invoice: InvoiceCreate,
    store: StoreDep,
) -> InvoiceOut:
    global next_id
    invoice_id = next_id
    next_id += 1
    entry = {"id": invoice_id, **invoice.model_dump()}
    store[invoice_id] = entry
    return InvoiceOut(**entry)


@app.get("/invoices/{invoice_id}", response_model=InvoiceOut)
async def read_invoice(
    invoice_id: int,
    store: StoreDep,
) -> InvoiceOut:
    if invoice_id not in store:
        raise HTTPException(status_code=status.HTTP_404_NOT_FOUND, detail="Invoice not found")
    return InvoiceOut(**store[invoice_id])


@app.get("/invoices", response_model=list[InvoiceOut])
async def list_invoices(
    store: StoreDep,
    skip: int = 0,
    limit: int = 10,
) -> list[InvoiceOut]:
    items = list(store.values())[skip : skip + limit]
    return [InvoiceOut(**item) for item in items]


@app.patch("/invoices/{invoice_id}", response_model=InvoiceOut)
async def update_invoice(
    invoice_id: int,
    update: InvoiceUpdate,
    store: StoreDep,
) -> InvoiceOut:
    if invoice_id not in store:
        raise HTTPException(status_code=status.HTTP_404_NOT_FOUND, detail="Invoice not found")
    stored = store[invoice_id]
    update_data = update.model_dump(exclude_unset=True)
    stored.update(update_data)
    store[invoice_id] = stored
    return InvoiceOut(**stored)


@app.delete("/invoices/{invoice_id}", status_code=status.HTTP_204_NO_CONTENT)
async def delete_invoice(
    invoice_id: int,
    store: StoreDep,
):
    if invoice_id not in store:
        raise HTTPException(status_code=status.HTTP_404_NOT_FOUND, detail="Invoice not found")
    del store[invoice_id]
    return None

## 5. TestClient tests

Plain pytest functions with `assert` — no running server, no network.

In [12]:
client = TestClient(app)


def test_create_invoice():
    response = client.post(
        "/invoices",
        json={"invoice_number": "INV-001", "vendor": "Acme Corp", "total": 1234.56},
    )
    assert response.status_code == 201
    body = response.json()
    assert body["invoice_number"] == "INV-001"
    assert body["vendor"] == "Acme Corp"
    assert body["total"] == 1234.56
    assert "id" in body


def test_read_invoice():
    create = client.post(
        "/invoices",
        json={"invoice_number": "INV-002", "vendor": "Widgets Inc", "total": 99.00},
    )
    invoice_id = create.json()["id"]
    response = client.get(f"/invoices/{invoice_id}")
    assert response.status_code == 200
    assert response.json()["invoice_number"] == "INV-002"


def test_read_invoice_not_found():
    response = client.get("/invoices/99999")
    assert response.status_code == 404
    assert "detail" in response.json()


def test_list_invoices_with_query_params():
    for i in range(3):
        client.post(
            "/invoices",
            json={
                "invoice_number": f"INV-10{i}",
                "vendor": f"Vendor {i}",
                "total": 10.0 * (i + 1),
            },
        )
    response = client.get("/invoices?skip=1&limit=2")
    assert response.status_code == 200
    body = response.json()
    assert len(body) == 2


def test_update_invoice_partial():
    create = client.post(
        "/invoices",
        json={"invoice_number": "INV-003", "vendor": "Old Vendor", "total": 50.0},
    )
    invoice_id = create.json()["id"]
    response = client.patch(
        f"/invoices/{invoice_id}",
        json={"total": 75.0},
    )
    assert response.status_code == 200
    assert response.json()["total"] == 75.0
    assert response.json()["vendor"] == "Old Vendor"


def test_delete_invoice():
    create = client.post(
        "/invoices",
        json={"invoice_number": "INV-DELETE", "vendor": "Temp", "total": 5.0},
    )
    invoice_id = create.json()["id"]
    response = client.delete(f"/invoices/{invoice_id}")
    assert response.status_code == 204
    response = client.get(f"/invoices/{invoice_id}")
    assert response.status_code == 404

## 6. Run the tests

```bash
python -m pytest -q
```

Expected: all 6 tests pass.

In [13]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "-x"],
    capture_output=True,
    text=True,
)

print(result.stdout)

if result.stderr:
    print(result.stderr)

if result.returncode == 5:
    print("No tests were collected. Check your test files and working directory.")
elif result.returncode != 0:
    print(f"Tests failed with exit code {result.returncode}.")
else:
    print("All tests passed.")

........                                                                 [100%]
8 passed in 0.75s

All tests passed.


## 7. Verify automatic docs

Start the server and visit:
- `http://127.0.0.1:8000/docs` — Swagger UI
- `http://127.0.0.1:8000/openapi.json` — raw schema

```bash
uvicorn <module>:app --reload
```

No extra code needed — FastAPI generates both from your type annotations.

In [14]:
openapi = app.openapi()
paths = list(openapi["paths"].keys())
print(f"Registered paths: {sorted(paths)}")
print(f"Title: {openapi['info']['title']}")
print(f"Version: {openapi['info']['version']}")

Registered paths: ['/invoices', '/invoices/{invoice_id}']
Title: Mini Invoice Tracker
Version: 0.1.0


## 8. Completion exercises

Do these by hand. No autocomplete.

In [15]:
# EXERCISE 1: Add a `GET /health` route that returns {"status": "ok"}
# EXERCISE 2: Add a dependency that validates an API key header (401 if missing/wrong)
# EXERCISE 3: Add `response_model_exclude_unset=True` to PATCH and verify unset fields are omitted
# EXERCISE 4: Write a TestClient test that posts invalid JSON (negative total) and asserts 422

print("Exercises ready — implement by hand.")

Exercises ready — implement by hand.
